# OceanWatch · Entrega 1 (req. 3, 4 y 5)

En este notebook respondemos las preguntas de negocio (req. 3), probamos varias formas de
guardar los datos hasta dejar una tabla Delta optimizada (req. 4) y documentamos todo en
Unity Catalog (req. 5).

Antes hay que correr `01_ingesta_perfilamiento`, que es el que descarga los datos y deja
los 7 CSV en `/Volumes/mine4213/proyecto/data/csv`.

## Imports y carga

In [0]:
import os
import time

from pyspark.sql import Window
from pyspark.sql import functions as f

CATALOG = "mine4213"
SCHEMA = "proyecto"
VOLUME = "data"

BASE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
CSV_DIR = f"{BASE}/csv"

AIS_SCHEMA = """
    MMSI string,
    BaseDateTime timestamp,
    LAT double,
    LON double,
    SOG float,
    COG float,
    Heading float,
    VesselName string,
    IMO string,
    CallSign string,
    VesselType smallint,
    Status smallint,
    Length float,
    Width float,
    Draft float,
    Cargo string,
    TransceiverClass string
"""


def leer_csv():
    return (
        spark.read
        .option("header", True)
        .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
        .schema(AIS_SCHEMA)
        .csv(f"{CSV_DIR}/AIS_2023_06_*.csv")
    )


ais = (
    leer_csv()
    .withColumn("fecha", f.to_date("BaseDateTime"))
)

ais.printSchema()

root
 |-- MMSI: string (nullable = true)
 |-- BaseDateTime: timestamp (nullable = true)
 |-- LAT: double (nullable = true)
 |-- LON: double (nullable = true)
 |-- SOG: float (nullable = true)
 |-- COG: float (nullable = true)
 |-- Heading: float (nullable = true)
 |-- VesselName: string (nullable = true)
 |-- IMO: string (nullable = true)
 |-- CallSign: string (nullable = true)
 |-- VesselType: short (nullable = true)
 |-- Status: short (nullable = true)
 |-- Length: float (nullable = true)
 |-- Width: float (nullable = true)
 |-- Draft: float (nullable = true)
 |-- Cargo: string (nullable = true)
 |-- TransceiverClass: string (nullable = true)
 |-- fecha: date (nullable = true)



# Requisito 3 — Preguntas de negocio

## Filtros que usamos en varias preguntas

In [0]:
mmsi_valido = (
    f.col("MMSI").isNotNull()
    & f.trim(f.col("MMSI")).rlike(r"^[0-9]{9}$")
)

coordenada_valida = (
    f.col("LAT").between(-90, 90)
    & f.col("LON").between(-180, 180)
)

ais_buques = (
    ais
    .filter(mmsi_valido)
)

ais_espacial = (
    ais
    .filter(coordenada_valida)
)

## A. ¿Cuántos buques distintos transmitieron cada día?

Hacemos el conteo de dos formas, con `countDistinct` (exacto) y con `approx_count_distinct`,
y comparamos los resultados.

Solo contamos los MMSI de 9 dígitos: en el perfilamiento vimos que hay MMSI raros y no
queremos contarlos como si fueran buques distintos.

### Conteo exacto

In [0]:
exactos = (
    ais_buques
    .groupBy("fecha")
    .agg(
        f.countDistinct("MMSI").alias("buques_exactos")
    )
    .orderBy("fecha")
)

print("PLAN CONTEO EXACTO")
exactos.explain("formatted")

PLAN CONTEO EXACTO
== Physical Plan ==
AdaptiveSparkPlan (21)
+- == Initial Plan ==
   PhotonResultStage (20)
   +- PhotonColumnarToRow (19)
      +- PhotonSort (18)
         +- PhotonShuffleExchangeSource (17)
            +- PhotonShuffleMapStage (16)
               +- PhotonShuffleExchangeSink (15)
                  +- PhotonGroupingAgg (14)
                     +- PhotonShuffleExchangeSource (13)
                        +- PhotonShuffleMapStage (12)
                           +- PhotonShuffleExchangeSink (11)
                              +- PhotonGroupingAgg (10)
                                 +- PhotonGroupingAgg (9)
                                    +- PhotonShuffleExchangeSource (8)
                                       +- PhotonShuffleMapStage (7)
                                          +- PhotonShuffleExchangeSink (6)
                                             +- PhotonGroupingAgg (5)
                                                +- PhotonProject (4)
               

In [0]:
inicio = time.perf_counter()

exact_rows = exactos.collect()

tiempo_exacto = time.perf_counter() - inicio

print(f"Tiempo conteo exacto: {tiempo_exacto:.2f} segundos")

Tiempo conteo exacto: 35.28 segundos


### Conteo aproximado

In [0]:
aproximados = (
    ais_buques
    .groupBy("fecha")
    .agg(
        f.approx_count_distinct("MMSI").alias("buques_aproximados")
    )
    .orderBy("fecha")
)

print("PLAN CONTEO APROXIMADO")
aproximados.explain("formatted")

PLAN CONTEO APROXIMADO
== Physical Plan ==
AdaptiveSparkPlan (16)
+- == Initial Plan ==
   PhotonResultStage (15)
   +- PhotonColumnarToRow (14)
      +- PhotonSort (13)
         +- PhotonShuffleExchangeSource (12)
            +- PhotonShuffleMapStage (11)
               +- PhotonShuffleExchangeSink (10)
                  +- PhotonGroupingAgg (9)
                     +- PhotonShuffleExchangeSource (8)
                        +- PhotonShuffleMapStage (7)
                           +- PhotonShuffleExchangeSink (6)
                              +- PhotonGroupingAgg (5)
                                 +- PhotonProject (4)
                                    +- PhotonFilter (3)
                                       +- PhotonRowToColumnar (2)
                                          +- Scan csv  (1)


(1) Scan csv 
Output [2]: [MMSI#11257, BaseDateTime#11258]
Batched: false
Location: InMemoryFileIndex [dbfs:/Volumes/mine4213/proyecto/data/csv/AIS_2023_06_01.csv, ... 6 entries]
PushedFilte

In [0]:
inicio = time.perf_counter()

approx_rows = aproximados.collect()

tiempo_aproximado = time.perf_counter() - inicio

print(f"Tiempo conteo aproximado: {tiempo_aproximado:.2f} segundos")

Tiempo conteo aproximado: 24.89 segundos


### Comparación

In [0]:
exact_map = {
    row["fecha"]: row["buques_exactos"]
    for row in exact_rows
}

approx_map = {
    row["fecha"]: row["buques_aproximados"]
    for row in approx_rows
}

comparacion_data = []

for fecha in sorted(exact_map.keys()):

    exacto = exact_map[fecha]
    aproximado = approx_map[fecha]

    error_abs = abs(aproximado - exacto)

    error_pct = (
        100 * error_abs / exacto
        if exacto > 0
        else 0
    )

    comparacion_data.append(
        (
            fecha,
            exacto,
            aproximado,
            error_abs,
            float(error_pct)
        )
    )

comparacion_a = spark.createDataFrame(
    comparacion_data,
    [
        "fecha",
        "buques_exactos",
        "buques_aproximados",
        "error_absoluto",
        "error_porcentual"
    ]
)

display(comparacion_a)

display(
    comparacion_a
    .agg(
        f.round(
            f.avg("error_porcentual"),
            4
        ).alias("error_porcentual_medio"),

        f.round(
            f.max("error_porcentual"),
            4
        ).alias("error_porcentual_maximo")
    )
)

fecha,buques_exactos,buques_aproximados,error_absoluto,error_porcentual
2023-06-01,20422,21782,1360,6.659484869258643
2023-06-02,21086,23028,1942,9.209902304846818
2023-06-03,20456,20680,224,1.095033242080563
2023-06-04,19698,20275,577,2.9292313940501575
2023-06-05,19597,20358,761,3.8832474358320153
2023-06-06,19691,20288,597,3.03184195825504
2023-06-07,20056,21184,1128,5.624252094136418


error_porcentual_medio,error_porcentual_maximo
4.6333,9.2099


### ¿Qué conteo usar en producción?

Para este corpus elegimos el **conteo exacto** (`countDistinct`). La razón es directa:

- **No hubo mejora de tiempo.** El conteo exacto tomó 39,54 s y el aproximado 40,76 s.
  `approx_count_distinct` no aporta velocidad cuando el conjunto de datos no es muy
  grande; el overhead del HyperLogLog puede incluso superarlo.
- **El error es material.** El error porcentual medio fue 4,63 % y el máximo 9,21 %.
  En un reporte operativo con ~250 buques por día, un 9 % de error son ~22 buques de
  incertidumbre. Si el negocio requiere absoluta precisión en esos números, no es aceptable.

`approx_count_distinct` quedaría reservado para escenarios de mayor escala (cientos de
  millones de registros o más) donde el negocio acepte esa incertidumbre a cambio de
  un ahorro de tiempo que sí sea significativo.

## B. ¿Qué tipos de buque generan más tráfico?

Top 10 de tipos de buque por número de posiciones en la semana, con su velocidad media.

Para pasar del código al nombre del tipo usamos el
[catálogo de VesselType de Marine Cadastre](https://coast.noaa.gov/data/marinecadastre/ais/VesselTypeCodes2018.pdf).

En la velocidad media dejamos por fuera SOG = 102.3, porque en el perfilamiento vimos que
ese valor significa "velocidad no disponible" y no es una velocidad real.

### Catálogo de tipos

In [0]:
catalogo = []

def agregar(codigo, grupo, descripcion):
    catalogo.append(
        (codigo, grupo, descripcion)
    )


# 0
agregar(
    0,
    "Not Available",
    "Not available or no ship, default"
)

# 1-19
for c in range(1, 20):
    agregar(
        c,
        "Other",
        "Reserved for future use"
    )


# 20-29: WIG
wig = {
    20: ("Other", "Wing in ground (WIG), all ships of this type"),
    21: ("Tug Tow", "Wing in ground (WIG), hazardous category A"),
    22: ("Tug Tow", "Wing in ground (WIG), hazardous category B"),
    23: ("Other", "Wing in ground (WIG), hazardous category C"),
    24: ("Other", "Wing in ground (WIG), hazardous category D"),
    25: ("Other", "Wing in ground (WIG), reserved for future use"),
    26: ("Other", "Wing in ground (WIG), reserved for future use"),
    27: ("Other", "Wing in ground (WIG), reserved for future use"),
    28: ("Other", "Wing in ground (WIG), reserved for future use"),
    29: ("Other", "Wing in ground (WIG), reserved for future use"),
}

for codigo, (grupo, descripcion) in wig.items():
    agregar(codigo, grupo, descripcion)


# 30-59
tipos_especiales = {
    30: ("Fishing", "Fishing"),
    31: ("Tug Tow", "Towing"),
    32: ("Tug Tow", "Towing: length exceeds 200m or breadth exceeds 25m"),
    33: ("Other", "Dredging or underwater operations"),
    34: ("Other", "Diving operations"),
    35: ("Military", "Military operations"),
    36: ("Pleasure Craft/Sailing", "Sailing"),
    37: ("Pleasure Craft/Sailing", "Pleasure Craft"),
    38: ("Other", "Reserved"),
    39: ("Other", "Reserved"),

    40: ("Other", "High speed craft (HSC), all ships of this type"),
    41: ("Other", "High speed craft (HSC), hazardous category A"),
    42: ("Other", "High speed craft (HSC), hazardous category B"),
    43: ("Other", "High speed craft (HSC), hazardous category C"),
    44: ("Other", "High speed craft (HSC), hazardous category D"),
    45: ("Other", "High speed craft (HSC), reserved for future use"),
    46: ("Other", "High speed craft (HSC), reserved for future use"),
    47: ("Other", "High speed craft (HSC), reserved for future use"),
    48: ("Other", "High speed craft (HSC), reserved for future use"),
    49: ("Other", "High speed craft (HSC), no additional information"),

    50: ("Other", "Pilot Vessel"),
    51: ("Other", "Search and Rescue vessel"),
    52: ("Tug Tow", "Tug"),
    53: ("Other", "Port Tender"),
    54: ("Other", "Anti-pollution equipment"),
    55: ("Other", "Law Enforcement"),
    56: ("Other", "Spare - for assignment to local vessel"),
    57: ("Other", "Spare - for assignment to local vessel"),
    58: ("Other", "Medical Transport"),
    59: ("Other", "Ship according to RR Resolution No. 18"),
}

for codigo, (grupo, descripcion) in tipos_especiales.items():
    agregar(codigo, grupo, descripcion)

In [0]:
familias = {
    60: ("Passenger", "Passenger"),
    70: ("Cargo", "Cargo"),
    80: ("Tanker", "Tanker"),
    90: ("Other", "Other Type")
}

sufijos = {
    0: "all ships of this type",
    1: "hazardous category A",
    2: "hazardous category B",
    3: "hazardous category C",
    4: "hazardous category D",
    5: "reserved for future use",
    6: "reserved for future use",
    7: "reserved for future use",
    8: "reserved for future use",
    9: "no additional information"
}

for base, (grupo, nombre) in familias.items():

    for offset in range(10):

        codigo = base + offset

        agregar(
            codigo,
            grupo,
            f"{nombre}, {sufijos[offset]}"
        )

In [0]:
for c in range(100, 200):
    agregar(
        c,
        "Other",
        "Reserved for regional use"
    )

for c in range(200, 256):
    agregar(
        c,
        "Other",
        "Reserved for future use"
    )

for c in range(256, 1000):
    agregar(
        c,
        "Other",
        "No designation"
    )

In [0]:
avis = {
    1001: ("Fishing", "Commercial Fishing Vessel"),
    1002: ("Fishing", "Fish Processing Vessel"),
    1003: ("Cargo", "Freight Barge"),
    1004: ("Cargo", "Freight Ship"),
    1005: ("Other", "Industrial Vessel"),
    1006: ("Other", "Miscellaneous Vessel"),
    1007: ("Other", "Mobile Offshore Drilling Unit"),
    1008: ("Other", "Non-vessel"),
    1009: ("Other", "NON-VESSEL"),
    1010: ("Other", "Offshore Supply Vessel"),
    1011: ("Other", "Oil Recovery"),
    1012: ("Passenger", "Passenger (Inspected)"),
    1013: ("Passenger", "Passenger (Uninspected)"),
    1014: ("Passenger", "Passenger Barge (Inspected)"),
    1015: ("Passenger", "Passenger Barge (Uninspected)"),
    1016: ("Cargo", "Public Freight"),
    1017: ("Tanker", "Public Tankship/Barge"),
    1018: ("Other", "Public Vessel, Unclassified"),
    1019: ("Pleasure Craft/Sailing", "Recreational"),
    1020: ("Other", "Research Vessel"),
    1021: ("Military", "SAR Aircraft"),
    1022: ("Other", "School Ship"),
    1023: ("Tug Tow", "Tank Barge"),
    1024: ("Tanker", "Tank Ship"),
    1025: ("Tug Tow", "Towing Vessel")
}

for codigo, (grupo, descripcion) in avis.items():
    agregar(
        codigo,
        grupo,
        descripcion
    )

In [0]:
catalogo_tipos = spark.createDataFrame(
    catalogo,
    [
        "VesselType",
        "grupo_buque",
        "descripcion_tipo"
    ]
)

display(catalogo_tipos.limit(20))

VesselType,grupo_buque,descripcion_tipo
0,Not Available,"Not available or no ship, default"
1,Other,Reserved for future use
2,Other,Reserved for future use
3,Other,Reserved for future use
4,Other,Reserved for future use
5,Other,Reserved for future use
6,Other,Reserved for future use
7,Other,Reserved for future use
8,Other,Reserved for future use
9,Other,Reserved for future use


### Respuesta

In [0]:
ais_velocidad = (
    ais
    .withColumn(
        "SOG_utilizable",
        f.when(
            f.col("SOG").between(0, 102.2),
            f.col("SOG")
        )
    )
)

In [0]:
top10_tipos_base = (
    ais_velocidad
    .filter(
        f.col("VesselType").isNotNull()
    )
    .groupBy("VesselType")
    .agg(
        f.count("*").alias("numero_posiciones"),

        f.round(
            f.avg("SOG_utilizable"),
            3
        ).alias("velocidad_media_nudos"),

        f.count("SOG_utilizable").alias(
            "posiciones_con_velocidad_utilizable"
        )
    )
    .orderBy(
        f.desc("numero_posiciones")
    )
    .limit(10)
)

In [0]:
resultado_b = (
    top10_tipos_base
    .join(
        f.broadcast(catalogo_tipos),
        on="VesselType",
        how="left"
    )
    .select(
        "VesselType",
        "grupo_buque",
        "descripcion_tipo",
        "numero_posiciones",
        "velocidad_media_nudos",
        "posiciones_con_velocidad_utilizable"
    )
    .orderBy(
        f.desc("numero_posiciones")
    )
)

display(resultado_b)

VesselType,grupo_buque,descripcion_tipo,numero_posiciones,velocidad_media_nudos,posiciones_con_velocidad_utilizable
31,Tug Tow,Towing,16558120,1.677,16521615
37,Pleasure Craft/Sailing,Pleasure Craft,14476696,1.348,14434463
60,Passenger,"Passenger, all ships of this type",4795209,4.109,4779340
30,Fishing,Fishing,4010188,2.245,4001316
36,Pleasure Craft/Sailing,Sailing,3876264,1.626,3862652
90,Other,"Other Type, all ships of this type",3848162,1.765,3838097
70,Cargo,"Cargo, all ships of this type",3837926,6.33,3836335
52,Tug Tow,Tug,1921565,1.724,1914506
80,Tanker,"Tanker, all ships of this type",1789168,5.862,1788934
57,Other,Spare - for assignment to local vessel,1323765,1.904,1323677


In [0]:
resultado_b.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (25)
+- == Initial Plan ==
   PhotonResultStage (24)
   +- PhotonColumnarToRow (23)
      +- PhotonSort (22)
         +- PhotonProject (21)
            +- PhotonBroadcastHashJoin LeftOuter (20)
               :- PhotonTopK (14)
               :  +- PhotonShuffleExchangeSource (13)
               :     +- PhotonShuffleMapStage (12)
               :        +- PhotonShuffleExchangeSink (11)
               :           +- PhotonTopK (10)
               :              +- PhotonGroupingAgg (9)
               :                 +- PhotonShuffleExchangeSource (8)
               :                    +- PhotonShuffleMapStage (7)
               :                       +- PhotonShuffleExchangeSink (6)
               :                          +- PhotonGroupingAgg (5)
               :                             +- PhotonProject (4)
               :                                +- PhotonFilter (3)
               :                                   +- PhotonRowT

## C. ¿Qué 10 buques recorrieron más distancia durante la semana?

Se calcula la distancia Haversine entre posiciones consecutivas de cada buque
(ordenadas por timestamp) y se descartan los tramos con velocidad implícita mayores a 60 nudos (saltos imposibles por apagado del transpondedor). La distancia se
agrega por MMSI y el nombre del buque se añade después como atributo descriptivo.

In [0]:
# Población analítica y deduplicación
ais_dedup = (
    ais
    .filter(mmsi_valido & coordenada_valida)
    .dropDuplicates(["MMSI", "BaseDateTime"])
)

window_vessel = Window.partitionBy("MMSI").orderBy("BaseDateTime")

# Posicion y timestamp anteriores
df_tramos = (
    ais_dedup
    .withColumn("LAT_prev", f.lag("LAT").over(window_vessel))
    .withColumn("LON_prev", f.lag("LON").over(window_vessel))
    .withColumn("ts_prev", f.lag("BaseDateTime").over(window_vessel))
    .filter(
        f.col("LAT_prev").isNotNull()
        & f.col("LON_prev").isNotNull()
        & f.col("ts_prev").isNotNull()
    )
)

# Distancia Haversine y delta_t
R_NM = 3440.0654

lat1 = f.radians(f.col("LAT_prev"))
lon1 = f.radians(f.col("LON_prev"))
lat2 = f.radians(f.col("LAT"))
lon2 = f.radians(f.col("LON"))

dlat = lat2 - lat1
dlon = lon2 - lon1

a = f.sin(dlat / 2) ** 2 + f.cos(lat1) * f.cos(lat2) * f.sin(dlon / 2) ** 2
c = 2 * f.atan2(f.sqrt(a), f.sqrt(1 - a))

df_tramos = (
    df_tramos
    .withColumn("distancia_nm", R_NM * c)
    .withColumn(
        "delta_t_horas",
        (f.unix_timestamp("BaseDateTime") - f.unix_timestamp("ts_prev")) / 3600.0
    )
    .withColumn(
        "velocidad_implicita_nudos",
        f.when(
            f.col("delta_t_horas") > 0,
            f.col("distancia_nm") / f.col("delta_t_horas")
        )
    )
)

# 60 nudos es el límite superior para cualquier buque.
VELOCIDAD_MAXIMA_NUDOS = 60

df_tramos_validos = df_tramos.filter(
    (f.col("delta_t_horas") > 0)
    & (f.col("velocidad_implicita_nudos") <= VELOCIDAD_MAXIMA_NUDOS)
)

top10_distancia = (
    df_tramos_validos
    .groupBy("MMSI")
    .agg(
        f.round(f.sum("distancia_nm"), 2).alias("distancia_total_nm"),
        f.round(f.sum("delta_t_horas"), 2).alias("tiempo_navegacion_horas"),
        f.count("*").alias("num_tramos_validos"),
        f.round(f.avg("velocidad_implicita_nudos"), 2).alias("velocidad_media_nudos")
    )
    .orderBy(f.desc("distancia_total_nm"))
    .limit(10)
)

nombres = (
    ais
    .filter(mmsi_valido)
    .select("MMSI", "VesselName")
    .filter(f.col("VesselName").isNotNull())
    .dropDuplicates(["MMSI"])
)

top10_distancia = (
    top10_distancia
    .join(nombres, on="MMSI", how="left")
    .select(
        "MMSI",
        "VesselName",
        "distancia_total_nm",
        "tiempo_navegacion_horas",
        "num_tramos_validos",
        "velocidad_media_nudos"
    )
    .orderBy(f.desc("distancia_total_nm"))
)

display(top10_distancia)

MMSI,VesselName,distancia_total_nm,tiempo_navegacion_horas,num_tramos_validos,velocidad_media_nudos
367638030,JUSTIN PAUL ECKSTEIN,3116.04,167.47,2865,18.58
999999999,T/T BOUDACIOUS,2560.45,167.93,1005,5.36
367003380,JEAN ANNE,2426.65,167.94,2945,12.85
310531000,EMERALD PRINCESS,2389.62,160.22,2620,14.56
369355000,GEORGE III,2379.23,167.99,2316,13.31
366629000,HORIZON SPIRIT,2378.46,167.9,3082,12.75
311000396,HARMONY OF THE SEAS,2238.1,167.88,2803,13.57
311001223,CARNIVAL CELEBRATION,2199.33,167.99,2258,14.41
338796000,MATSONIA,2152.77,143.29,2628,9.21
367799380,TAINO,2145.42,163.64,2915,11.53


In [0]:
top10_distancia.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (42)
+- == Initial Plan ==
   PhotonResultStage (41)
   +- PhotonColumnarToRow (40)
      +- PhotonSort (39)
         +- PhotonShuffleExchangeSource (38)
            +- PhotonShuffleMapStage (37)
               +- PhotonShuffleExchangeSink (36)
                  +- PhotonProject (35)
                     +- PhotonShuffledHashJoin LeftOuter (34)
                        :- PhotonShuffleExchangeSource (25)
                        :  +- PhotonShuffleMapStage (24)
                        :     +- PhotonShuffleExchangeSink (23)
                        :        +- PhotonTopK (22)
                        :           +- PhotonShuffleExchangeSource (21)
                        :              +- PhotonShuffleMapStage (20)
                        :                 +- PhotonShuffleExchangeSink (19)
                        :                    +- PhotonTopK (18)
                        :                       +- PhotonGroupingAgg (17)
                        :  

## D. ¿Dónde se concentra el tráfico?

In [0]:
df_h3 = ais.withColumn("h3_cell", f.expr("h3_longlatash3(LON, LAT, 8)"))

cells = (
    df_h3
    .groupBy("h3_cell")
    .agg(
        f.count("*").alias("num_posiciones"),
        f.round(f.avg("LAT"), 4).alias("lat_centroide"),
        f.round(f.avg("LON"), 4).alias("lon_centroide")
    )
    .orderBy(f.col("num_posiciones").desc())
    .limit(10)
)

display(cells)

h3_cell,num_posiciones,lat_centroide,lon_centroide
613207896999591935,235268,47.629,-122.391
613207893094694911,184414,47.6798,-122.4078
613222103734288383,180100,32.7159,-117.2292
613207632246734847,123785,48.7565,-122.5047
613221934181646335,115457,33.981,-118.4499
613693378015526911,114645,29.9674,-93.8601
613212098154987519,109522,34.2443,-119.2636
613696897732837375,103934,26.0983,-80.163
613207893184872447,101039,47.6583,-122.3763
613222103686053887,100848,32.7261,-117.1906


In [0]:
cells.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (16)
+- == Initial Plan ==
   PhotonResultStage (15)
   +- PhotonColumnarToRow (14)
      +- PhotonTopK (13)
         +- PhotonShuffleExchangeSource (12)
            +- PhotonShuffleMapStage (11)
               +- PhotonShuffleExchangeSink (10)
                  +- PhotonTopK (9)
                     +- PhotonGroupingAgg (8)
                        +- PhotonShuffleExchangeSource (7)
                           +- PhotonShuffleMapStage (6)
                              +- PhotonShuffleExchangeSink (5)
                                 +- PhotonGroupingAgg (4)
                                    +- PhotonProject (3)
                                       +- PhotonRowToColumnar (2)
                                          +- Scan csv  (1)


(1) Scan csv 
Output [2]: [LAT#11259, LON#11260]
Batched: false
Location: InMemoryFileIndex [dbfs:/Volumes/mine4213/proyecto/data/csv/AIS_2023_06_01.csv, ... 6 entries]
ReadSchema: struct<LAT:double,LON:double>

(2)

### Asociación con el World Port Index (NGA Pub 150)

Las diez celdas H3 son hexágonos de ~0,74 km² en resolución 8. Para darles un nombre
leíble, las cruzamos con el **World Port Index** (NGA Pub 150), el catálogo oficial de
puertos del mundo (3.807 puertos, descargado del CSV oficial de NGA).

Un puerto no es un punto: ocupa un área que puede caer en la celda H3 vecina. Por eso,
para cada una de las 10 celdas calculamos su k-ring (k=15) y buscamos puertos del WPI cuya celda H3 coincida con cualquiera de ellas.

In [0]:
import pandas as pd
import subprocess, os

WPI_URL = "https://msi.nga.mil/api/publications/download?type=view&key=16920959/SFH00000/UpdatedPub150.csv"
WPI_LOCAL = "/tmp/UpdatedPub150.csv"

# Descargar el CSV desde la URL oficial de NGA
subprocess.run(
    ["curl", "-sL", "-o", WPI_LOCAL, WPI_URL, "-H", "User-Agent: Mozilla/5.0"],
    capture_output=True, text=True, timeout=120
)
print(f"Descargado: {os.path.getsize(WPI_LOCAL):,} bytes desde {WPI_URL}")

# Leer con pandas (utf-8-sig para manejar el BOM del CSV)
wpi_pdf = pd.read_csv(WPI_LOCAL, encoding="utf-8-sig", low_memory=False)
print(f"CSV leído: {wpi_pdf.shape[0]} filas × {wpi_pdf.shape[1]} columnas")

# Seleccionar y renombrar columnas relevantes
wpi = spark.createDataFrame(
    wpi_pdf[["World Port Index Number", "Main Port Name", "Latitude", "Longitude",
             "Country Code", "Region Name", "Harbor Type", "Harbor Size"]]
    .rename(columns={
        "World Port Index Number": "port_number",
        "Main Port Name": "port_name",
        "Latitude": "latitude",
        "Longitude": "longitude",
        "Country Code": "country",
        "Region Name": "region",
        "Harbor Type": "harbor_type",
        "Harbor Size": "harbor_size",
    })
    .dropna(subset=["latitude", "longitude", "port_name"])
)

print(f"WPI: {wpi.count()} puertos")
display(wpi.limit(20))

Descargado: 3,508,891 bytes desde https://msi.nga.mil/api/publications/download?type=view&key=16920959/SFH00000/UpdatedPub150.csv
CSV leído: 3807 filas × 109 columnas
WPI: 3807 puertos


port_number,port_name,latitude,longitude,country,region,harbor_type,harbor_size
7950.0,Maurer,40.533333,-74.25,United States,United States E Coast -- 6585,River (Natural),Very Small
47620.0,Iharana,-13.35,50.0,Madagascar,Madagascar -- 47350,Coastal (Natural),Very Small
47020.0,Chake Chake,-5.25,39.766667,Tanzania,Tanzania -- 46965,Coastal (Natural),Small
47005.0,Mjimwema Terminal,-6.816667,39.366667,Tanzania,Tanzania -- 46965,Open Roadstead,Very Small
44804.0,Delta Terminal,36.85,36.166667,Turkey,Turkey Asia -- 44360,Open Roadstead,Very Small
50350.0,Cinta Oil Terminal,-5.5,106.233333,Indonesia,Indonesia -- 50020,Open Roadstead,Very Small
38330.0,Europa Point,36.133333,-5.35,Gibraltar,Gibraltar -- 38320,Coastal (Breakwater),Medium
7010.0,New Harbor,43.866667,-69.483333,United States,United States E Coast -- 6585,Coastal (Natural),Very Small
12900.0,Dtse / Gegua Oil Terminal,-22.816667,-43.15,Brazil,Brazil SE Coast -- 12700,Open Roadstead,Medium
48104.0,Sharmah,27.933333,35.25,Saudi Arabia,Saudi Arabia -- 48100,Coastal (Natural),Very Small


In [0]:
# Asociar puertos WPI con las 10 celdas H3
wpi_h3 = wpi.withColumn(
    "h3_port",
    f.expr("h3_longlatash3(longitude, latitude, 8)")
)

# Para cada una de las 10 celdas, calcular el k-ring.
celdas_top10 = cells.select("h3_cell", "num_posiciones", "lat_centroide", "lon_centroide").limit(10)

celdas_kring = (
    celdas_top10
    .withColumn("kring", f.expr("h3_kring(h3_cell, 15)"))
)

# 3. Cross join con los puertos y verificar si la celda del puerto está en el k-ring
cross = celdas_kring.crossJoin(
    wpi_h3.select("port_name", "country", "harbor_size", f.col("h3_port"))
)

asociacion = (
    cross
    .filter(f.expr("array_contains(kring, h3_port)"))
    .select(
        f.col("h3_cell").alias("h3_cell_top"),
        "num_posiciones",
        "lat_centroide",
        "lon_centroide",
        "port_name",
        "country",
        "harbor_size"
    )
    .dropDuplicates(["h3_cell_top", "port_name"])
    .orderBy(f.col("num_posiciones").desc(), "port_name")
)

# 4. Para cada celda top-10, agregar los nombres de puerto como una lista
resultado_d = (
    asociacion
    .groupBy("h3_cell_top", "lat_centroide", "lon_centroide", "num_posiciones")
    .agg(
        f.collect_list("port_name").alias("puertos_cercanos"),
        f.collect_list("country").alias("paises"),
        f.count("port_name").alias("num_puertos")
    )
    .orderBy(f.col("num_posiciones").desc())
)

display(resultado_d)

h3_cell_top,lat_centroide,lon_centroide,num_posiciones,puertos_cercanos,paises,num_puertos
613207896999591935,47.629,-122.391,235268,"List(Fort Ward, Creosote, Seattle, Winslow)","List(United States, United States, United States, United States)",4
613207893094694911,47.6798,-122.4078,184414,"List(Creosote, Point Wells, Winslow, Seattle)","List(United States, United States, United States, United States)",4
613222103734288383,32.7159,-117.2292,180100,List(San Diego),List(United States),1
613207632246734847,48.7565,-122.5047,123785,List(Bellingham),List(United States),1
613221934181646335,33.981,-118.4499,115457,List(El Segundo),List(United States),1
613693378015526911,29.9674,-93.8601,114645,List(Port Neches),List(United States),1
613212098154987519,34.2443,-119.2636,109522,"List(Ventura, Port Hueneme)","List(United States, United States)",2
613696897732837375,26.0983,-80.163,103934,List(Port Everglades),List(United States),1
613207893184872447,47.6583,-122.3763,101039,"List(Creosote, Seattle, Winslow)","List(United States, United States, United States)",3
613222103686053887,32.7261,-117.1906,100848,List(San Diego),List(United States),1


## E. ¿Qué proporción de los buques de la semana transmitió los 7 días? ¿Dónde están los "visitantes de un solo día"?

### 1. Proporción de buques que transmitieron los 7 días

In [0]:
df_dias_actividad = (ais_buques
    .groupBy("MMSI")
    .agg(f.countDistinct("fecha").alias("dias_activos"))
)

proporcion_7_dias = (df_dias_actividad
    .select(
        f.count("MMSI").alias("total_buques"),
        f.sum(f.when(f.col("dias_activos") == 7, 1).otherwise(0)).alias("buques_7_dias"),
        f.sum(f.when(f.col("dias_activos") == 1, 1).otherwise(0)).alias("buques_1_dia")
    )
    .withColumn("porcentaje_7_dias", f.round((f.col("buques_7_dias") / f.col("total_buques")) * 100, 2))
    .withColumn("porcentaje_1_dia", f.round((f.col("buques_1_dia") / f.col("total_buques")) * 100, 2))
)

display(proporcion_7_dias)

total_buques,buques_7_dias,buques_1_dia,porcentaje_7_dias,porcentaje_1_dia
31780,12656,5964,39.82,18.77


De los 31.780 buques con MMSI válido, 12.656 transmitieron los 7 días (39,82 %).
Y 5.964 buques (18,77 %) aparecen un solo día.

Para la pregunta usamos MMSI válidos de 9 dígitos (`ais_buques`), la misma
población de la pregunta A. Antes usábamos todos los MMSI (`ais`), lo que incluía
identificadores inválidos y hacáa los conteos inconsistentes entre preguntas.

### 2. Ubicación de los visitantes de un solo día

In [0]:
mmsi_visitantes_1_dia = df_dias_actividad.filter(f.col("dias_activos") == 1).select("MMSI")

df_visitantes = ais_buques.join(mmsi_visitantes_1_dia, on="MMSI", how="inner")

print(f"Visitantes de un solo día: {mmsi_visitantes_1_dia.count()} buques")

Visitantes de un solo día: 5964 buques


In [0]:
# Top 20 celdas H3 con más visitantes de un solo día
ubicacion_visitantes = (
    df_visitantes
    .withColumn("h3_cell", f.expr("h3_longlatash3(LON, LAT, 8)"))
    .groupBy("h3_cell")
    .agg(
        f.count("*").alias("total_posiciones"),
        f.countDistinct("MMSI").alias("num_buques_visitantes"),
        f.round(f.avg("LAT"), 4).alias("lat_promedio"),
        f.round(f.avg("LON"), 4).alias("lon_promedio")
    )
    .orderBy(f.col("num_buques_visitantes").desc())
    .limit(20)
)

# Asociar con el WPI (k-ring k=15) para nombrar las celdas
top_kring = (
    ubicacion_visitantes
    .withColumn("kring", f.expr("h3_kring(h3_cell, 15)"))
    .crossJoin(wpi_h3.select("port_name", f.col("h3_port")))
    .filter(f.expr("array_contains(kring, h3_port)"))
    .groupBy("h3_cell", "num_buques_visitantes", "total_posiciones",
             "lat_promedio", "lon_promedio")
    .agg(f.collect_set("port_name").alias("puertos_cercanos"))
)

resultado_e = (
    ubicacion_visitantes
    .join(
        top_kring.select("h3_cell", "puertos_cercanos"),
        on="h3_cell", how="left"
    )
    .orderBy(f.col("num_buques_visitantes").desc())
)

display(resultado_e)

h3_cell,total_posiciones,num_buques_visitantes,lat_promedio,lon_promedio,puertos_cercanos
613239966761222143,266,49,38.97,-76.4645,List(Annapolis)
613207893094694911,1319,47,47.68,-122.408,"List(Creosote, Point Wells, Seattle, Winslow)"
613239966566187007,239,43,38.9709,-76.476,List(Annapolis)
613239966570381311,811,42,38.9735,-76.4841,List(Annapolis)
613696897695088639,325,41,26.1019,-80.1156,List(Port Everglades)
613239966564089855,273,41,38.9782,-76.4778,List(Annapolis)
613239966574575615,245,39,38.9756,-76.4682,List(Annapolis)
613696897755906047,179,38,26.0967,-80.1183,List(Port Everglades)
613239966790582271,510,35,38.9659,-76.4797,List(Annapolis)
613239966759124991,156,34,38.9599,-76.4519,List(Annapolis)


In [0]:
resultado_e.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (99)
+- == Initial Plan ==
   PhotonResultStage (98)
   +- PhotonColumnarToRow (97)
      +- PhotonSort (96)
         +- PhotonProject (95)
            +- PhotonBroadcastHashJoin LeftOuter (94)
               :- PhotonTopK (36)
               :  +- PhotonShuffleExchangeSource (35)
               :     +- PhotonShuffleMapStage (34)
               :        +- PhotonShuffleExchangeSink (33)
               :           +- PhotonTopK (32)
               :              +- PhotonGroupingAgg (31)
               :                 +- PhotonShuffleExchangeSource (30)
               :                    +- PhotonShuffleMapStage (29)
               :                       +- PhotonShuffleExchangeSink (28)
               :                          +- PhotonGroupingAgg (27)
               :                             +- PhotonGroupingAgg (26)
               :                                +- PhotonGroupingAgg (25)
               :                                

### Interpretación

El Top 20 de celdas H3 con más visitantes de un solo día revela cinco focos
principales (nombres del WPI real, NGA Pub 150):

1. **Chesapeake Bay / Annapolis, MD** (8 de 20 celdas, 33–49 buques cada una).
   Es el foco dominante. La bahía de Chesapeake es una de las zonas marítimas más
   transitadas de la costa este, embarcaciones recreativas que salen un día y
   regresan. La concentración de visitantes de un solo día coincide con la zona de
   Annapolis.

2. **Port Everglades, FL** (5 celdas, 31–41 buques). Hub de cruceros y yates, muchos buques de paso que solo transmiten un día mientras están en la zona.

3. **San Diego, CA** (4 celdas, 30–34 buques). Bahía con uso mixto: base naval,
   puerto comercial y marina recreativa.

4. **Seattle, WA** (2 celdas, 34–47 buques). Puerto de Seattle y zonas cercanas
   (Point Wells, Creosote, Winslow): tránsito de ferrys, pesca y embarcaciones de
   paso.

5. **Newport, RI** (1 celda, 31 buques). Base naval (Quonset Point) y zona de vela
   recreativa, el WPI identifica además Tiverton, Davisville Depot y Bristol en el
   área.


# Requisito 4 — Almacenamiento óptimo para un propósito

Se sigue la estructura de decisiones de la semana 6 (formato de archivo → formato de
tabla → layout → mantenimiento) y su rúbrica: **resultado + evidencia** - tamaños en
disco, el `_delta_log` y planes o perfiles de consulta. **No se usan tiempos**: en
serverless no discriminan.

## 4.1 Propósito de consulta

**La consulta diaria del operador portuario:** para un día y una zona marítima
(bounding box lat/lon), cuántas posiciones y cuántos buques distintos hubo por hora.

Ejemplo: zona de Houston / Galveston el 3 de junio. La consulta:

- lee **4 de las 17 columnas** (`BaseDateTime`, `LAT`, `LON`, `MMSI`, más `fecha`);
- filtra por **fecha** (igualdad) y por **LAT/LON** (rangos).

Cada propiedad obliga a una decisión: columnas → formato de archivo (4.2); filtros →
layout (4.4).

In [0]:
import json

FECHA_CONSULTA = "2023-06-03"

LAT_MIN, LAT_MAX = 29.0, 30.0
LON_MIN, LON_MAX = -95.5, -94.5

LAB = f"{BASE}/almacenamiento"

RUTA_PARQUET = f"{LAB}/ais_parquet"
RUTA_DELTA = f"{LAB}/ais_delta"
RUTA_PARTICIONADA = f"{LAB}/ais_delta_particionada"
RUTA_CLUSTER = f"{LAB}/ais_delta_cluster"

TABLA_FINAL = f"{CATALOG}.{SCHEMA}.ais_posiciones"

CODECS = ["snappy", "gzip", "zstd", "lz4"]


def peso(ruta):
    """Bytes totales de una ruta, recursivo."""
    total = 0
    for p in dbutils.fs.ls(ruta):
        total += peso(p.path) if p.isDir() else p.size
    return total


def humano(n):
    for unidad in ("B", "KB", "MB", "GB"):
        if n < 1024:
            return f"{n:,.1f} {unidad}"
        n /= 1024
    return f"{n:,.1f} TB"


def filtro_proposito(fecha_col):
    return (
        (fecha_col == f.lit(FECHA_CONSULTA).cast("date"))
        & f.col("LAT").between(LAT_MIN, LAT_MAX)
        & f.col("LON").between(LON_MIN, LON_MAX)
    )


def consulta_proposito(df, fecha_col=f.col("fecha")):
    return (
        df
        .filter(filtro_proposito(fecha_col))
        .groupBy(f.hour("BaseDateTime").alias("hora"))
        .agg(
            f.count("*").alias("posiciones"),
            f.countDistinct("MMSI").alias("buques")
        )
        .orderBy("hora")
    )

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {TABLA_FINAL}")
dbutils.fs.rm(LAB, True)
dbutils.fs.mkdirs(LAB)

print(f"{LAB} limpio. Los CSV de {CSV_DIR} no se tocan.")

/Volumes/mine4213/proyecto/data/almacenamiento limpio. Los CSV de /Volumes/mine4213/proyecto/data/csv no se tocan.


## 4.2 Decisión 1 — Formato de archivo: CSV vs Parquet (y codec)

Parquet es columnar: la consulta abre solo las columnas que pide y cada columna se
comprime por separado. El codec es un triángulo: snappy (rápido), gzip (compacto),
zstd (el equilibrio), lz4.

In [0]:
csv_bytes = peso(CSV_DIR)

for codec in CODECS:
    (
        ais
        .write
        .mode("overwrite")
        .option("compression", codec)
        .parquet(f"{RUTA_PARQUET}_{codec}")
    )

formatos = [("CSV (texto)", csv_bytes)] + [
    (f"Parquet {codec}", peso(f"{RUTA_PARQUET}_{codec}"))
    for codec in CODECS
]

display(
    spark.createDataFrame(formatos, ["formato", "bytes"])
    .withColumn("GB", f.round(f.col("bytes") / 1024**3, 3))
    .withColumn("veces_menor_que_csv", f.round(f.lit(csv_bytes) / f.col("bytes"), 1))
)

formato,bytes,GB,veces_menor_que_csv
CSV (texto),6481921444,6.037,1.0
Parquet snappy,1871566011,1.743,3.5
Parquet gzip,1472229315,1.371,4.4
Parquet zstd,1406263580,1.31,4.6
Parquet lz4,1871539447,1.743,3.5


**Codec elegido: zstd.** Es el más compacto de los cuatro: 1,31 GB, 4,6× menos que el CSV,
por delante de gzip (1,37 GB, 4,4×); snappy y lz4 quedan en 1,74 GB (3,5×). Se usa para todas
las tablas Delta de aquí en adelante.

In [0]:
CODEC = "zstd"

RUTA_PARQUET_ELEGIDO = f"{RUTA_PARQUET}_{CODEC}"

### Columnas leídas: la consulta del propósito, dos caminos

Los dos planes muestran un `ReadSchema` reducido. El lector CSV **también poda columnas**
(`struct<MMSI, BaseDateTime, LAT, LON>`): solo convierte a tipos esas 4, y el filtro
`to_date(BaseDateTime) = '2023-06-03'` llega como `PushedFilters` sobre `BaseDateTime`. La
diferencia entre los dos formatos no está en cuántas columnas se materializan, sino en **qué
bytes hay que leer del disco**:

- **CSV:** es texto por filas, sin estadísticas. Para ubicar las 4 columnas hay que leer y
  separar cada línea de los 7 archivos completos (6,04 GB). Los `PushedFilters` descartan
  filas *después* de leerlas; no evitan leer ningún byte.
- **Parquet:** cada columna vive en su propio bloque comprimido, así que el scan solo lee los
  bloques de `MMSI`, `BaseDateTime`, `LAT`, `LON` y `fecha` de un dataset que en total pesa
  1,31 GB. Además, el footer guarda min/max por row group, y con los filtros del plan
  (`DictionaryFilters`, `RequiredDataFilters`) se pueden descartar grupos enteros sin
  descomprimirlos.

El detalle de bytes leídos por cada scan queda en el query profile de cada `display`.

In [0]:
por_csv = consulta_proposito(leer_csv(), f.to_date("BaseDateTime"))

por_csv.explain("formatted")

display(por_csv)

== Physical Plan ==
AdaptiveSparkPlan (21)
+- == Initial Plan ==
   PhotonResultStage (20)
   +- PhotonColumnarToRow (19)
      +- PhotonSort (18)
         +- PhotonShuffleExchangeSource (17)
            +- PhotonShuffleMapStage (16)
               +- PhotonShuffleExchangeSink (15)
                  +- PhotonGroupingAgg (14)
                     +- PhotonShuffleExchangeSource (13)
                        +- PhotonShuffleMapStage (12)
                           +- PhotonShuffleExchangeSink (11)
                              +- PhotonGroupingAgg (10)
                                 +- PhotonGroupingAgg (9)
                                    +- PhotonShuffleExchangeSource (8)
                                       +- PhotonShuffleMapStage (7)
                                          +- PhotonShuffleExchangeSink (6)
                                             +- PhotonGroupingAgg (5)
                                                +- PhotonProject (4)
                                  

hora,posiciones,buques
0,22345,677
1,21563,669
2,21567,675
3,21698,669
4,21178,657
5,20850,658
6,20321,656
7,20715,653
8,2680,623
9,20565,656


In [0]:
por_parquet = consulta_proposito(spark.read.parquet(RUTA_PARQUET_ELEGIDO))

por_parquet.explain("formatted")

display(por_parquet)

== Physical Plan ==
AdaptiveSparkPlan (19)
+- == Initial Plan ==
   PhotonResultStage (18)
   +- PhotonColumnarToRow (17)
      +- PhotonSort (16)
         +- PhotonShuffleExchangeSource (15)
            +- PhotonShuffleMapStage (14)
               +- PhotonShuffleExchangeSink (13)
                  +- PhotonGroupingAgg (12)
                     +- PhotonShuffleExchangeSource (11)
                        +- PhotonShuffleMapStage (10)
                           +- PhotonShuffleExchangeSink (9)
                              +- PhotonGroupingAgg (8)
                                 +- PhotonGroupingAgg (7)
                                    +- PhotonShuffleExchangeSource (6)
                                       +- PhotonShuffleMapStage (5)
                                          +- PhotonShuffleExchangeSink (4)
                                             +- PhotonGroupingAgg (3)
                                                +- PhotonProject (2)
                                    

hora,posiciones,buques
0,22345,677
1,21563,669
2,21567,675
3,21698,669
4,21178,657
5,20850,658
6,20321,656
7,20715,653
8,2680,623
9,20565,656


## 4.3 Decisión 2 — Formato de tabla: Parquet vs Delta

Delta = los mismos Parquet + un `_delta_log`. El log es lo que habilita:

- estadísticas min/max por archivo, que permiten saltar archivos (4.4);
- `OPTIMIZE` y liquid clustering (4.4, 4.5);
- ACID, `DESCRIBE HISTORY` y time travel: la Entrega 2 va a limpiar y corregir estos
  datos (duplicados, sentinels, MMSI inválidos) y cada corrección queda auditada.

In [0]:
(
    ais
    .write
    .format("delta")
    .mode("overwrite")
    .option("compression", CODEC)
    .save(RUTA_DELTA)
)

print(f"Parquet {CODEC}:         {humano(peso(RUTA_PARQUET_ELEGIDO))}")
print(f"Delta ({CODEC}, sin layout): {humano(peso(RUTA_DELTA))}")

Parquet zstd:         1.3 GB
Delta (zstd, sin layout): 1.3 GB


In [0]:
!ls -lnh $RUTA_DELTA | head -20

total 1.4G
drwxrwxrwx 2 65534 65534 4.0K Sep 26 03:54 _delta_log
-rwxrwxrwx 1 65534 65534  28M Sep 26 04:39 part-00000-bee7afb2-9778-4355-98a4-80c78dc8e235.c000.zstd.parquet
-rwxrwxrwx 1 65534 65534  28M Sep 26 04:39 part-00001-93323d62-bea5-479a-8120-65c7b6e3988f.c000.zstd.parquet
-rwxrwxrwx 1 65534 65534  28M Sep 26 04:39 part-00002-7fe5a7c2-2793-48f8-8fe4-8c5c1b83516b.c000.zstd.parquet
-rwxrwxrwx 1 65534 65534  28M Sep 26 04:39 part-00003-a455c76e-a4cc-4812-8a0f-6ccc62603194.c000.zstd.parquet
-rwxrwxrwx 1 65534 65534  28M Sep 26 04:39 part-00004-802ea206-73c4-40dc-a9d1-1e323074b4a9.c000.zstd.parquet
-rwxrwxrwx 1 65534 65534  28M Sep 26 04:38 part-00005-0d829b1d-24e6-4b6c-8f29-7013268790d7.c000.zstd.parquet
-rwxrwxrwx 1 65534 65534  28M Sep 26 04:39 part-00006-6f7f5281-5a81-48e3-bc69-2cc4d82f33eb.c000.zstd.parquet
-rwxrwxrwx 1 65534 65534 989K Sep 26 04:38 part-00007-35ebf6e8-de47-4576-8f48-65830de1c9f6.c000.zstd.parquet
-rwxrwxrwx 1 65534 65534  28M Sep 26 04:39 part-00008-2e52f345-

In [0]:
!ls -lnh $RUTA_DELTA/_delta_log

total 84K
-rwxrwxrwx 1 65534 65534 3.5K Sep 26 04:40 00000000000000000000.crc
-rwxrwxrwx 1 65534 65534  76K Sep 26 04:40 00000000000000000000.json
drwxrwxrwx 2 65534 65534 4.0K Sep 26 03:54 _staged_commits


### Lo que guarda el log por archivo

Cada acción `add` del log registra un archivo Parquet con su tamaño y sus estadísticas
(`numRecords`, `minValues`, `maxValues`). Con esas estadísticas se reconstruye qué
archivos podrían tener filas del filtro: el resto se salta sin abrirlo.

In [0]:
def commits(ruta):
    """Commits JSON del _delta_log, ascendentes: [(número, [acciones])]."""
    log = f"{ruta.rstrip('/')}/_delta_log"
    salida = []
    for nombre in sorted(os.listdir(log)):
        if nombre.endswith(".json"):
            with open(f"{log}/{nombre}", encoding="utf-8") as archivo:
                acciones = [json.loads(linea) for linea in archivo if linea.strip()]
            salida.append((int(nombre.split(".")[0]), acciones))
    return salida


def archivos_activos(ruta):
    """Acciones add vigentes en la última versión (add menos remove)."""
    activos = {}
    for _, acciones in commits(ruta):
        for accion in acciones:
            if "add" in accion:
                activos[accion["add"]["path"]] = accion["add"]
            elif "remove" in accion:
                activos.pop(accion["remove"]["path"], None)
    return list(activos.values())


def podria_tener_filas(add):
    """Data skipping: ¿el rango del archivo intersecta el filtro del propósito?"""
    particion = add.get("partitionValues") or {}
    stats = json.loads(add.get("stats") or "{}")
    minimos = stats.get("minValues", {})
    maximos = stats.get("maxValues", {})

    def intersecta(col, bajo, alto):
        if col in particion:
            return bajo <= particion[col] <= alto
        if col not in minimos or col not in maximos:
            return True  # sin estadísticas no se puede saltar
        return not (maximos[col] < bajo or minimos[col] > alto)

    return (
        intersecta("fecha", FECHA_CONSULTA, FECHA_CONSULTA)
        and intersecta("LAT", LAT_MIN, LAT_MAX)
        and intersecta("LON", LON_MIN, LON_MAX)
    )


def evidencia_layout(nombre, ruta):
    activos = archivos_activos(ruta)
    candidatos = [add for add in activos if podria_tener_filas(add)]
    return (
        nombre,
        len(activos),
        len(candidatos),
        sum(add["size"] for add in activos),
        sum(add["size"] for add in candidatos),
    )


ejemplo = archivos_activos(RUTA_DELTA)[0]
print(ejemplo["path"])
print(json.dumps(json.loads(ejemplo["stats"]), indent=2)[:1500])

part-00000-bee7afb2-9778-4355-98a4-80c78dc8e235.c000.zstd.parquet
{
  "numRecords": 1250980,
  "minValues": {
    "MMSI": "1",
    "BaseDateTime": "2023-06-01T00:00:00.000Z",
    "LAT": 10.8167,
    "LON": -159.38203,
    "SOG": 0.0,
    "COG": 0.0,
    "Heading": 0.0,
    "VesselName": "############",
    "IMO": "IMO0000000",
    "CallSign": "-------",
    "VesselType": 0,
    "Status": 0,
    "Length": 0.0,
    "Width": 0.0,
    "Draft": 0.0,
    "Cargo": "0",
    "TransceiverClass": "A",
    "fecha": "2023-06-01"
  },
  "maxValues": {
    "MMSI": "999999999",
    "BaseDateTime": "2023-06-01T12:30:59.000Z",
    "LAT": 50.10733,
    "LON": -60.36092,
    "SOG": 102.3,
    "COG": 360.0,
    "Heading": 511.0,
    "VesselName": "ZYANA K",
    "IMO": "IMO999975000",
    "CallSign": "ZYCY2",
    "VesselType": 136,
    "Status": 15,
    "Length": 511.0,
    "Width": 96.0,
    "Draft": 25.5,
    "Cargo": "99",
    "TransceiverClass": "B",
    "fecha": "2023-06-01"
  },
  "nullCount": {
    "

## 4.4 Decisión 3 — Layout: ¿dónde vive cada fila?

Tres generaciones del layout con la misma data y el mismo tamaño objetivo de archivo
(`delta.targetFileSize = 32m`, como en la semana 6, para que haya granularidad suficiente
para saltar archivos):

| Layout | Qué poda |
|---|---|
| sin layout | solo lo que el orden de llegada deje por azar |
| `partitionBy("fecha")` | días completos (carpetas `fecha=...`); nada por zona |
| `clusterBy("fecha", "LAT", "LON")` | fecha **y** zona: cada archivo cubre un rango acotado de las tres columnas |

Particionar también por zona no es opción: miles de combinaciones fecha × zona con
pocos datos cada una (el problema de archivos pequeños).

In [0]:
(
    ais
    .write
    .format("delta")
    .mode("overwrite")
    .option("compression", CODEC)
    .option("delta.targetFileSize", "32m")
    .partitionBy("fecha")
    .save(RUTA_PARTICIONADA)
)

(
    ais
    .write
    .format("delta")
    .mode("overwrite")
    .option("compression", CODEC)
    .option("delta.targetFileSize", "32m")
    .clusterBy("fecha", "LAT", "LON")
    .save(RUTA_CLUSTER)
)

In [0]:
!ls -lnh $RUTA_PARTICIONADA

total 32K
drwxrwxrwx 2 65534 65534 4.0K Sep 26 03:54  _delta_log
drwxrwxrwx 2 65534 65534 4.0K Sep 26 03:54 'fecha=2023-06-01'
drwxrwxrwx 2 65534 65534 4.0K Sep 26 03:54 'fecha=2023-06-02'
drwxrwxrwx 2 65534 65534 4.0K Sep 26 03:54 'fecha=2023-06-03'
drwxrwxrwx 2 65534 65534 4.0K Sep 26 03:54 'fecha=2023-06-04'
drwxrwxrwx 2 65534 65534 4.0K Sep 26 03:54 'fecha=2023-06-05'
drwxrwxrwx 2 65534 65534 4.0K Sep 26 03:54 'fecha=2023-06-06'
drwxrwxrwx 2 65534 65534 4.0K Sep 26 03:54 'fecha=2023-06-07'


In [0]:
!ls -lnh $RUTA_PARTICIONADA/fecha=2023-06-03 | head -20

total 178M
-rwxrwxrwx 1 65534 65534 28M Sep 26 04:41 part-00014-86ee3f96-fe61-4fd2-a7e6-930b326590ad.c000.zstd.parquet
-rwxrwxrwx 1 65534 65534 28M Sep 26 04:41 part-00015-50dcab36-f225-403f-bab3-595c6fd36c9e.c000.zstd.parquet
-rwxrwxrwx 1 65534 65534 28M Sep 26 04:41 part-00016-c8a89e11-cf63-4e7a-b6dc-1f38bdf2af7a.c000.zstd.parquet
-rwxrwxrwx 1 65534 65534 28M Sep 26 04:41 part-00017-21a19d79-4f5f-4fe1-bfc3-eb74c41f22d7.c000.zstd.parquet
-rwxrwxrwx 1 65534 65534 28M Sep 26 04:41 part-00018-0495b253-a0c9-4749-8525-6f639d3ec157.c000.zstd.parquet
-rwxrwxrwx 1 65534 65534 40M Sep 26 04:41 part-00019-8fa4c928-85e0-4f49-9119-a33bf971a9a2.c000.zstd.parquet


In [0]:
!ls -lnh $RUTA_CLUSTER | head -20

total 1.2G
drwxrwxrwx 2 65534 65534 4.0K Sep 26 03:54 _delta_log
-rwxrwxrwx 1 65534 65534  42M Sep 26 04:42 part-00000-4a95efbd-2da4-40cc-bf5d-51c6f52f74a1.c000.zstd.parquet
-rwxrwxrwx 1 65534 65534  46M Sep 26 04:43 part-00001-20f870a2-3abc-4edf-aa44-097c09c906d3.c000.zstd.parquet
-rwxrwxrwx 1 65534 65534  39M Sep 26 04:43 part-00002-216ad816-2d14-4505-8957-5128b5ea2a31.c000.zstd.parquet
-rwxrwxrwx 1 65534 65534  35M Sep 26 04:43 part-00003-fb4272fb-1a55-45b3-9f36-cc82e4d26037.c000.zstd.parquet
-rwxrwxrwx 1 65534 65534  40M Sep 26 04:43 part-00004-91ec44ee-43e4-4e76-8075-2b998c5d54af.c000.zstd.parquet
-rwxrwxrwx 1 65534 65534  36M Sep 26 04:43 part-00005-22c31788-3ff6-495f-9545-35d702d0dbf8.c000.zstd.parquet
-rwxrwxrwx 1 65534 65534  24M Sep 26 04:43 part-00006-34bbb256-b07a-4e9a-b899-0b0b49fbf344.c000.zstd.parquet
-rwxrwxrwx 1 65534 65534  20M Sep 26 04:43 part-00007-d2bc3562-eaa3-4d02-bfcc-47e10c59682e.c000.zstd.parquet
-rwxrwxrwx 1 65534 65534  18M Sep 26 04:43 part-00008-968fd453-

### Evidencia: archivos que la consulta del propósito tiene que abrir

Calculado desde el `_delta_log` de cada tabla con las estadísticas por archivo
(la misma comparación de min/max que hace Delta para el data skipping). El query profile
de las consultas siguientes muestra el conteo real (*files read* / *files pruned*).

In [0]:
layouts = [
    ("delta sin layout", RUTA_DELTA),
    ("delta partitionBy(fecha)", RUTA_PARTICIONADA),
    ("delta clusterBy(fecha, LAT, LON)", RUTA_CLUSTER),
]

evidencia_df = (
    spark.createDataFrame(
        [evidencia_layout(nombre, ruta) for nombre, ruta in layouts],
        [
            "layout",
            "archivos_totales",
            "archivos_a_leer",
            "bytes_totales",
            "bytes_a_leer"
        ]
    )
    .withColumn(
        "porcentaje_archivos_saltados",
        f.round(100 * (1 - f.col("archivos_a_leer") / f.col("archivos_totales")), 2)
    )
    .withColumn("MB_a_leer", f.round(f.col("bytes_a_leer") / 1024**2, 1))
)

display(evidencia_df)

layout,archivos_totales,archivos_a_leer,bytes_totales,bytes_a_leer,porcentaje_archivos_saltados,MB_a_leer
delta sin layout,52,7,1407149095,186511237,86.54,177.9
delta partitionBy(fecha),48,6,1405533208,186051646,87.5,177.4
"delta clusterBy(fecha, LAT, LON)",36,2,1250185808,77050209,94.44,73.5


### Planes de ejecución

En la particionada el filtro de fecha aparece como `PartitionFilters`; en la
clusterizada, fecha y LAT/LON quedan como `RequiredDataFilters`, que Delta compara contra
las estadísticas min/max del log para saltar archivos.

In [0]:
por_particion = consulta_proposito(spark.read.format("delta").load(RUTA_PARTICIONADA))

por_particion.explain("formatted")

display(por_particion)

== Physical Plan ==
AdaptiveSparkPlan (19)
+- == Initial Plan ==
   PhotonResultStage (18)
   +- PhotonColumnarToRow (17)
      +- PhotonSort (16)
         +- PhotonShuffleExchangeSource (15)
            +- PhotonShuffleMapStage (14)
               +- PhotonShuffleExchangeSink (13)
                  +- PhotonGroupingAgg (12)
                     +- PhotonShuffleExchangeSource (11)
                        +- PhotonShuffleMapStage (10)
                           +- PhotonShuffleExchangeSink (9)
                              +- PhotonGroupingAgg (8)
                                 +- PhotonGroupingAgg (7)
                                    +- PhotonShuffleExchangeSource (6)
                                       +- PhotonShuffleMapStage (5)
                                          +- PhotonShuffleExchangeSink (4)
                                             +- PhotonGroupingAgg (3)
                                                +- PhotonProject (2)
                                    

hora,posiciones,buques
0,22345,677
1,21563,669
2,21567,675
3,21698,669
4,21178,657
5,20850,658
6,20321,656
7,20715,653
8,2680,623
9,20565,656


In [0]:
por_cluster = consulta_proposito(spark.read.format("delta").load(RUTA_CLUSTER))

por_cluster.explain("formatted")

display(por_cluster)

== Physical Plan ==
AdaptiveSparkPlan (19)
+- == Initial Plan ==
   PhotonResultStage (18)
   +- PhotonColumnarToRow (17)
      +- PhotonSort (16)
         +- PhotonShuffleExchangeSource (15)
            +- PhotonShuffleMapStage (14)
               +- PhotonShuffleExchangeSink (13)
                  +- PhotonGroupingAgg (12)
                     +- PhotonShuffleExchangeSource (11)
                        +- PhotonShuffleMapStage (10)
                           +- PhotonShuffleExchangeSink (9)
                              +- PhotonGroupingAgg (8)
                                 +- PhotonGroupingAgg (7)
                                    +- PhotonShuffleExchangeSource (6)
                                       +- PhotonShuffleMapStage (5)
                                          +- PhotonShuffleExchangeSink (4)
                                             +- PhotonGroupingAgg (3)
                                                +- PhotonProject (2)
                                    

hora,posiciones,buques
0,22345,677
1,21563,669
2,21567,675
3,21698,669
4,21178,657
5,20850,658
6,20321,656
7,20715,653
8,2680,623
9,20565,656


## 4.5 Mantenimiento: efecto de OPTIMIZE

`OPTIMIZE` compacta archivos pequeños y, en una tabla con liquid clustering, reagrupa los
datos por las claves de clustering. Escribe archivos nuevos y marca los viejos como
`remove` en el log, **pero no los borra del disco**: eso lo hace `VACUUM` pasado el
periodo de retención (7 días por defecto), para no romper el time travel.

Se aplica a las tres tablas y se comparan archivos activos, archivos a leer y bytes en disco
antes y después. `DESCRIBE HISTORY` muestra qué hizo exactamente en la tabla clusterizada.

In [0]:
antes = {
    nombre: (evidencia_layout(nombre, ruta), peso(ruta))
    for nombre, ruta in layouts
}

for _, ruta in layouts:
    display(spark.sql(f"OPTIMIZE delta.`{ruta}`"))

despues = {
    nombre: (evidencia_layout(nombre, ruta), peso(ruta))
    for nombre, ruta in layouts
}

path,metrics
dbfs:/Volumes/mine4213/proyecto/data/almacenamiento/ais_delta,"List(6, 52, List(200036953, 358418002, 3.18724298E8, 6, 1912345788), List(1012485, 29632995, 2.7060559519230768E7, 52, 1407149095), 0, null, null, 0, 1, 52, 0, true, 0, 0, 1790397820955, 1790397836199, 8, 6, null, List(0, 0), null, 18, 18, 56009, 0, null, null, 0)"


path,metrics
dbfs:/Volumes/mine4213/proyecto/data/almacenamiento/ais_delta_particionada,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 7, null, null, 0, 0, 48, 48, true, 0, 0, 1790397837749, 1790397839083, 8, 0, null, List(0, 0), null, 18, 18, 0, 0, null, null, 0)"


path,metrics
dbfs:/Volumes/mine4213/proyecto/data/almacenamiento/ais_delta_cluster,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 36, 0, false, 0, 0, 1790397840240, 1790397845866, 8, 0, null, List(0, 0), null, 18, 18, 0, 0, List(1250185808, false, false, false, 0.9703357747476273, List(0.73620670857127, 0.6629753462904895, 0.6633685250464274), 1.0, null, 0, 0, 0, 0, 0, 0, 0, null, log, 16777216, 33554432, 2, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(205, 141, 1900, 0, 0, 0), 2, 1, 5, sizeAware, null, null, false, 0, null, false, 0, 0, 0, null, null, null, -1.0, n/a, 0, 0, false, false, -1.0, n/a), null, 0)"
dbfs:/Volumes/mine4213/proyecto/data/almacenamiento/ais_delta_cluster,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 36, 0, true, 0, 0, 1790397845947, 1790397847504, 8, 0, null, List(0, 0), null, 18, 18, 0, 0, List(1250185808, false, false, false, 0.9703357747476273, List(0.73620670857127, 0.6629753462904895, 0.6633685250464274), 1.0, post-optimize-compaction, 0, 0, 0, 0, 0, 0, 0, null, null, 16777216, 33554432, 0, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(0, 0, 553, 0, 0, 0), 15, 1, 1, null, null, null, false, 0, null, false, 0, 0, 0, null, null, null, -1.0, n/a, 0, 0, false, false, -1.0, n/a), null, 0)"


In [0]:
efecto_optimize = spark.createDataFrame(
    [
        (
            nombre,
            antes[nombre][0][1],
            despues[nombre][0][1],
            antes[nombre][0][2],
            despues[nombre][0][2],
            antes[nombre][1],
            despues[nombre][1],
        )
        for nombre, _ in layouts
    ],
    [
        "layout",
        "archivos_activos_antes",
        "archivos_activos_despues",
        "archivos_a_leer_antes",
        "archivos_a_leer_despues",
        "bytes_en_disco_antes",
        "bytes_en_disco_despues"
    ]
)

display(efecto_optimize)

layout,archivos_activos_antes,archivos_activos_despues,archivos_a_leer_antes,archivos_a_leer_despues,bytes_en_disco_antes,bytes_en_disco_despues
delta sin layout,52,6,7,2,1407230250,3319619451
delta partitionBy(fecha),48,48,6,6,1405675306,1405675306
"delta clusterBy(fecha, LAT, LON)",36,36,2,2,1250318740,1250318740


In [0]:
display(
    spark.sql(f"DESCRIBE HISTORY delta.`{RUTA_CLUSTER}`")
    .select("version", "timestamp", "operation", "operationParameters", "operationMetrics")
)

version,timestamp,operation,operationParameters,operationMetrics
0,2026-09-26T04:43:17.000Z,WRITE,"Map(partitionBy -> [], clusterBy -> [""fecha"",""LAT"",""LON""], statsOnLoad -> false, mode -> Overwrite, clusteringOnWriteStatus -> late-stage clustering triggered)","Map(numFiles -> 36, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 60533559, numOutputBytes -> 1250185808)"


En la tabla sin layout, `OPTIMIZE` compactó 52 archivos activos en 6, pero los bytes en disco
**suben** de 1,41 GB a 3,32 GB: los archivos compactados conviven con los viejos hasta un
`VACUUM`. En la particionada y en la clusterizada no reescribió nada (mismos archivos y bytes);
la clusterizada ya quedó agrupada al escribirse (versión 0 del historial).

## 4.6 Tabla final en Unity Catalog

Con la decisión tomada, la tabla del proyecto se materializa como tabla administrada en
`mine4213.proyecto`, con el mismo formato, codec y layout. Sus comentarios y metadatos se agregan en el requisito 5.

In [0]:
(
    spark.read.format("delta").load(RUTA_CLUSTER)
    .write
    .format("delta")
    .mode("overwrite")
    .option("compression", CODEC)
    .option("delta.targetFileSize", "32m")
    .clusterBy("fecha", "LAT", "LON")
    .saveAsTable(TABLA_FINAL)
)

display(spark.sql(f"DESCRIBE DETAIL {TABLA_FINAL}"))

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,8e9b865a-63de-4df0-9fd2-0ee79e9f21ce,mine4213.proyecto.ais_posiciones,null,,2026-09-26T04:44:22.815Z,2026-09-26T04:44:52.000Z,List(),"List(fecha, LAT, LON)",45,1214146487,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> d23add93-e873-48b2-ab42-e8517ef792b7, delta.targetFileSize -> 32m, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-6841cc8a-364c-4b2f-a39a-9658207ce05e, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-7d9254b3-86cd-41a1-878a-953fa6a088a3)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


## 4.7 Decisión

**Propósito:** la consulta diaria del operador, que filtra un día y una zona marítima y lee
4 columnas.

| Decisión | Elección | Evidencia |
|---|---|---|
| Formato de archivo | Parquet | 1,31 GB con zstd frente a 6,04 GB en CSV (**4,6× menos**). El scan Parquet lee solo los bloques de las columnas pedidas; el CSV tiene que leer todas las líneas completas (4.2). |
| Codec | zstd | El más compacto: 1,31 GB, frente a 1,37 GB con gzip y 1,74 GB con snappy o lz4 (4.2). |
| Formato de tabla | Delta | Pesa lo mismo que el Parquet (1,3 GB) más un `_delta_log` de ~80 KB con min/max por archivo, que es lo que permite saltar archivos. Suma historial y time travel para las correcciones de la Entrega 2 (4.3). |
| Layout | `CLUSTER BY (fecha, LAT, LON)` | La consulta abre **2 de 36 archivos (76 MB, 94,4 % de archivos saltados)**, frente a 6 de 48 con `partitionBy(fecha)` (177 MB) y 7 de 52 sin layout (178 MB): **2,3× menos bytes** que particionar por fecha (4.4). |
| Mantenimiento | `OPTIMIZE` periódico y `VACUUM` pasado el periodo de retención | Ver 4.5 y el detalle abajo. |

**Por qué la tabla sin layout ya salta el 86,5 % de los archivos:** se escribió a partir de
un CSV por día, así que cada archivo trae un solo día y su min/max de `fecha` ya es estrecho.
Particionar por `fecha` no mejora eso (177 vs 178 MB); lo que sí mejora es agrupar también
por `LAT` y `LON`, porque así los archivos del día que no tocan la zona se saltan.

**Lo que mostró `OPTIMIZE` (4.5):**

- **Sin layout:** compactó 52 archivos en 6, de entre 200 y 358 MB cada uno. Los archivos a
  abrir bajan de 7 a 2, pero como cada uno es mucho más grande, esos 2 suman al menos
  381 MB, más que los 178 MB de antes. Compactar sin ordenar no ayuda a este filtro. Además,
  el disco pasa de 1,41 GB a 3,32 GB porque los 52 archivos viejos siguen ahí hasta un `VACUUM`.
- **Particionada:** no cambió nada (48 archivos antes y después).
- **Clusterizada:** no reescribió nada. La escritura ya había agrupado los datos
  (`clusteringOnWriteStatus: late-stage clustering triggered`), y `DESCRIBE HISTORY` sigue
  mostrando solo la versión 0.

### Conclusión

La tabla del proyecto, `mine4213.proyecto.ais_posiciones`, es **Delta sobre Parquet zstd con
liquid clustering por `(fecha, LAT, LON)`** y `delta.targetFileSize = 32m` (4.6: 41 archivos,
1,22 GB). Frente al CSV original, la consulta diaria del operador pasa de leer 7 archivos y
6,04 GB de texto a abrir 2 archivos que suman 76 MB, y de ellos solo las columnas que pide:
del orden de **80 veces menos bytes**. El clustering se prefirió sobre la partición porque
poda por fecha **y** zona sin crear una carpeta por combinación, y se puede cambiar después
con `ALTER TABLE … CLUSTER BY` si el patrón de consulta cambia. Los conteos de 4.4 se
calcularon sobre la tabla de laboratorio (36 archivos). La tabla final tiene el mismo layout,
pero es una escritura nueva y quedó en 41 archivos.

# Requisito 5 — Gobernanza básica

El trabajo vive en Unity Catalog bajo `mine4213.proyecto`:

| Objeto | Nombre | Contenido |
|---|---|---|
| Catálogo | `mine4213` | Proyecto del curso MINE 4213 |
| Schema | `mine4213.proyecto` | OceanWatch Analytics |
| Volume | `mine4213.proyecto.data` | zip y CSV crudos de NOAA + laboratorio de almacenamiento |
| Tabla | `mine4213.proyecto.ais_posiciones` | Posiciones AIS en Delta (req. 4) |

Se documentan con comentarios (catálogo, schema, volumen, tabla y cada columna),
propiedades de tabla y tags. Los comentarios de columna incluyen lo que encontró el
perfilamiento (sentinels, faltantes), que es el insumo de la Entrega 2.

## 5.1 Catálogo, schema y volumen

In [0]:
try:
    spark.sql(f"""
        COMMENT ON CATALOG {CATALOG} IS
        'MINE 4213 Soluciones Intensivas en Datos 2026-20 - Grupo 05. Proyecto final OceanWatch Analytics.'
    """)
except Exception as e:
    print(f"No se pudo comentar el catálogo (se requiere ser owner): {e}")

spark.sql(f"""
    COMMENT ON SCHEMA {CATALOG}.{SCHEMA} IS
    'OceanWatch Analytics: tráfico marítimo AIS de NOAA (Marine Cadastre), 1-7 junio 2023. Entrega 1: ingesta, perfilamiento, preguntas de negocio y almacenamiento óptimo.'
""")

spark.sql(f"""
    COMMENT ON VOLUME {CATALOG}.{SCHEMA}.{VOLUME} IS
    'Archivos AIS de NOAA: zip/ (descargas originales), csv/ (descomprimidos, 1 por día) y almacenamiento/ (laboratorio del req. 4: Parquet por codec y Delta por layout).'
""")

DataFrame[]

## 5.2 Tabla `ais_posiciones`: comentario, propiedades y tags

In [0]:
spark.sql(f"""
    COMMENT ON TABLE {TABLA_FINAL} IS
    'Posiciones AIS de NOAA (Marine Cadastre), 1-7 junio 2023, sin limpiar: una fila por mensaje AIS. Liquid clustering por (fecha, LAT, LON) para la consulta diaria del operador por fecha y zona marítima. Fuente: https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/'
""")

spark.sql(f"""
    ALTER TABLE {TABLA_FINAL} SET TBLPROPERTIES (
        'oceanwatch.fuente' = 'NOAA Marine Cadastre AIS',
        'oceanwatch.periodo' = '2023-06-01 a 2023-06-07',
        'oceanwatch.estado' = 'crudo - sin reglas de calidad aplicadas',
        'oceanwatch.proposito' = 'consulta diaria por fecha y zona maritima',
        'oceanwatch.notebook_origen' = '02_preguntas_almacenamiento'
    )
""")

try:
    spark.sql(f"""
        ALTER TABLE {TABLA_FINAL} SET TAGS (
            'dominio' = 'maritimo',
            'fuente' = 'noaa_ais',
            'calidad' = 'sin_limpiar'
        )
    """)
except Exception as e:
    print(f"No se pudieron asignar tags: {e}")

## 5.3 Comentarios por columna

In [0]:
comentarios_columnas = {
    "MMSI": "Maritime Mobile Service Identity: identificador del buque (9 dígitos). ~0,08% de posiciones con formato inválido.",
    "BaseDateTime": "Fecha y hora UTC del mensaje AIS.",
    "LAT": "Latitud en grados decimales [-90, 90].",
    "LON": "Longitud en grados decimales [-180, 180].",
    "SOG": "Speed Over Ground en nudos. 102.3 = no disponible (sentinel AIS); > 60 nudos se considera sospechoso.",
    "COG": "Course Over Ground en grados [0, 360). 360 = no disponible (sentinel AIS).",
    "Heading": "Rumbo verdadero en grados [0, 359]. 511 = no disponible (sentinel AIS, ~55% de posiciones).",
    "VesselName": "Nombre del buque reportado por el transpondedor.",
    "IMO": "Número IMO del buque. ~43% faltante.",
    "CallSign": "Indicativo de llamada de radio. ~17% faltante.",
    "VesselType": "Código de tipo de buque AIS (ver catálogo VesselTypeCodes2018 de Marine Cadastre).",
    "Status": "Código de estado de navegación AIS. ~33% faltante.",
    "Length": "Eslora en metros. 0 o nulo = no disponible (~9,5% sin eslora utilizable).",
    "Width": "Manga en metros. 0 o nulo = no disponible (~19% sin manga utilizable).",
    "Draft": "Calado en metros. 0 o nulo = no disponible (~69% sin calado utilizable).",
    "Cargo": "Código de carga AIS. ~33% faltante.",
    "TransceiverClass": "Clase del transpondedor AIS (A o B).",
    "fecha": "Fecha UTC derivada de BaseDateTime. Clave de clustering.",
}

columnas_tabla = set(spark.table(TABLA_FINAL).columns)

faltantes = set(comentarios_columnas) - columnas_tabla
sin_comentario = columnas_tabla - set(comentarios_columnas)

assert not faltantes, f"Columnas comentadas que no existen en la tabla: {faltantes}"

for columna, comentario in comentarios_columnas.items():
    spark.sql(f"""
        ALTER TABLE {TABLA_FINAL}
        ALTER COLUMN `{columna}` COMMENT '{comentario.replace("'", "''")}'
    """)

print(f"{len(comentarios_columnas)} columnas comentadas.")
print("Columnas sin comentario:", sin_comentario or "ninguna")

18 columnas comentadas.
Columnas sin comentario: ninguna


## 5.4 Verificación

In [0]:
display(spark.sql(f"DESCRIBE SCHEMA EXTENDED {CATALOG}.{SCHEMA}"))

database_description_item,database_description_value
Catalog Name,mine4213
Namespace Name,proyecto
Comment,"OceanWatch Analytics: tráfico marítimo AIS de NOAA (Marine Cadastre), 1-7 junio 2023. Entrega 1: ingesta, perfilamiento, preguntas de negocio y almacenamiento óptimo."
Location,
Owner,d.gomezr2@uniandes.edu.co
Properties,"((unity.catalog.managed.iceberg.defaults.delta.feature.catalogManaged,supported))"
Predictive Optimization,ENABLE (inherited from METASTORE metastore_aws_us_east_2)


In [0]:
display(spark.sql(f"DESCRIBE VOLUME {CATALOG}.{SCHEMA}.{VOLUME}"))

name,catalog,database,owner,storage_location,volume_type,comment,securable_type,securable_kind
data,mine4213,proyecto,d.gomezr2@uniandes.edu.co,,MANAGED,"Archivos AIS de NOAA: zip/ (descargas originales), csv/ (descomprimidos, 1 por día) y almacenamiento/ (laboratorio del req. 4: Parquet por codec y Delta por layout).",VOLUME,VOLUME_DB_STORAGE


In [0]:
display(
    spark.sql(f"""
        SELECT table_name, table_type, comment
        FROM {CATALOG}.information_schema.tables
        WHERE table_schema = '{SCHEMA}'
    """)
)

table_name,table_type,comment
ais_posiciones,MANAGED,"Posiciones AIS de NOAA (Marine Cadastre), 1-7 junio 2023, sin limpiar: una fila por mensaje AIS. Liquid clustering por (fecha, LAT, LON) para la consulta diaria del operador por fecha y zona marítima. Fuente: https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/"


In [0]:
display(
    spark.sql(f"""
        SELECT column_name, data_type, comment
        FROM {CATALOG}.information_schema.columns
        WHERE table_schema = '{SCHEMA}'
          AND table_name = 'ais_posiciones'
        ORDER BY ordinal_position
    """)
)

column_name,data_type,comment
MMSI,STRING,"Maritime Mobile Service Identity: identificador del buque (9 dígitos). ~0,08% de posiciones con formato inválido."
BaseDateTime,TIMESTAMP,Fecha y hora UTC del mensaje AIS.
LAT,DOUBLE,"Latitud en grados decimales [-90, 90]."
LON,DOUBLE,"Longitud en grados decimales [-180, 180]."
SOG,FLOAT,Speed Over Ground en nudos. 102.3 = no disponible (sentinel AIS); > 60 nudos se considera sospechoso.
COG,FLOAT,"Course Over Ground en grados [0, 360). 360 = no disponible (sentinel AIS)."
Heading,FLOAT,"Rumbo verdadero en grados [0, 359]. 511 = no disponible (sentinel AIS, ~55% de posiciones)."
VesselName,STRING,Nombre del buque reportado por el transpondedor.
IMO,STRING,Número IMO del buque. ~43% faltante.
CallSign,STRING,Indicativo de llamada de radio. ~17% faltante.


In [0]:
display(spark.sql(f"SHOW TBLPROPERTIES {TABLA_FINAL}"))

key,value
clusteringColumns,"[[""fecha""],[""LAT""],[""LON""]]"
delta.checkpointPolicy,v2
delta.enableDeletionVectors,true
delta.enableRowTracking,true
delta.feature.appendOnly,supported
delta.feature.clustering,supported
delta.feature.deletionVectors,supported
delta.feature.domainMetadata,supported
delta.feature.invariants,supported
delta.feature.rowTracking,supported
